In [0]:
# Databricks notebook:
# 03_state_transaction_batch_scoring
#
# Purpose:
# Load the approved Champion model, generate 2026-Q3
# state-level forecasts, and save them to a Delta table.

import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np

from mlflow import MlflowClient
from pyspark.sql import functions as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

SCORING_FEATURE_TABLE = (
    "workspace.gold.mart_ml_state_scoring_features"
)

REGISTERED_MODEL_NAME = (
    "workspace.ml."
    "phonepe_state_transaction_forecaster"
)

MODEL_ALIAS = "Champion"

FORECAST_TABLE = (
    "workspace.ml.state_transaction_forecasts"
)

EXPECTED_STATE_COUNT = 36

MODEL_FEATURES = [
    "state_slug",

    "quarter",
    "time_index",
    "quarter_sin",
    "quarter_cos",

    "total_transaction_count",
    "registered_user_count",
    "registered_merchant_count",

    "transactions_per_registered_user",
    "transactions_per_registered_merchant",

    "lag_1_transaction_count",
    "lag_2_transaction_count",
    "lag_3_transaction_count",
    "lag_4_transaction_count",

    "rolling_4q_transaction_mean",
    "rolling_4q_transaction_stddev",

    "transaction_yoy_growth_pct",

    "transaction_rank",
    "registered_user_rank",

    "transaction_qoq_growth_pct",
    "registered_user_qoq_growth_pct",
    "registered_merchant_qoq_growth_pct"
]

print("=" * 70)
print("PHONEPE STATE TRANSACTION BATCH SCORING")
print("=" * 70)

print(f"Feature table : {SCORING_FEATURE_TABLE}")
print(f"Model         : {REGISTERED_MODEL_NAME}")
print(f"Model alias   : {MODEL_ALIAS}")
print(f"Forecast table: {FORECAST_TABLE}")


# ============================================================
# 2. LOAD AND VALIDATE SCORING FEATURES
# ============================================================

scoring_spark_df = spark.table(
    SCORING_FEATURE_TABLE
)

scoring_summary = (
    scoring_spark_df
    .agg(
        F.count("*").alias("row_count"),

        F.countDistinct(
            "state_slug"
        ).alias("state_count"),

        F.countDistinct(
            "feature_period"
        ).alias("feature_period_count"),

        F.countDistinct(
            "target_period"
        ).alias("target_period_count"),

        F.min(
            "feature_period"
        ).alias("feature_period"),

        F.min(
            "target_period"
        ).alias("target_period")
    )
    .first()
)

missing_feature_expression = None

for feature_name in MODEL_FEATURES:

    current_expression = F.when(
        F.col(feature_name).isNull(),
        1
    ).otherwise(0)

    if missing_feature_expression is None:
        missing_feature_expression = (
            current_expression
        )
    else:
        missing_feature_expression = (
            missing_feature_expression
            + current_expression
        )

missing_feature_values = (
    scoring_spark_df
    .select(
        F.sum(
            missing_feature_expression
        ).alias("missing_feature_values")
    )
    .first()["missing_feature_values"]
)

print("\nScoring feature validation")
print(
    f"Rows          : "
    f"{scoring_summary['row_count']:,}"
)
print(
    f"States/UTs    : "
    f"{scoring_summary['state_count']:,}"
)
print(
    f"Feature period: "
    f"{scoring_summary['feature_period']}"
)
print(
    f"Target period : "
    f"{scoring_summary['target_period']}"
)
print(
    f"Missing values: "
    f"{missing_feature_values:,}"
)

assert (
    scoring_summary["row_count"]
    == EXPECTED_STATE_COUNT
)

assert (
    scoring_summary["state_count"]
    == EXPECTED_STATE_COUNT
)

assert scoring_summary["feature_period_count"] == 1
assert scoring_summary["target_period_count"] == 1
assert missing_feature_values == 0

print("Scoring feature validation passed.")


# ============================================================
# 3. LOAD CHAMPION MODEL
# ============================================================

mlflow.set_registry_uri("databricks-uc")

registry_client = MlflowClient(
    registry_uri="databricks-uc"
)

champion_model_version = (
    registry_client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME,
        alias=MODEL_ALIAS
    )
)

MODEL_VERSION = str(
    champion_model_version.version
)

CHAMPION_MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"@{MODEL_ALIAS}"
)

champion_model = mlflow.pyfunc.load_model(
    CHAMPION_MODEL_URI
)

print("\nChampion model loaded")
print(f"Model URI    : {CHAMPION_MODEL_URI}")
print(f"Model version: {MODEL_VERSION}")


# ============================================================
# 4. PREPARE MODEL INPUT
# ============================================================

scoring_pd = (
    scoring_spark_df
    .orderBy("state_slug")
    .toPandas()
)

missing_model_columns = sorted(
    set(MODEL_FEATURES)
    - set(scoring_pd.columns)
)

assert not missing_model_columns, (
    f"Missing model features: "
    f"{missing_model_columns}"
)

model_input_pd = (
    scoring_pd[
        MODEL_FEATURES
    ]
    .copy()
)

model_input_pd["state_slug"] = (
    model_input_pd["state_slug"]
    .astype(str)
)

for feature_name in MODEL_FEATURES:

    if feature_name != "state_slug":

        model_input_pd[feature_name] = (
            pd.to_numeric(
                model_input_pd[feature_name],
                errors="raise"
            )
        )

assert not model_input_pd.isnull().any().any()

numerical_input = (
    model_input_pd
    .drop(columns=["state_slug"])
    .to_numpy(dtype=float)
)

assert np.isfinite(numerical_input).all()

print("\nModel input validation passed.")
print(
    f"Model input rows    : "
    f"{len(model_input_pd):,}"
)
print(
    f"Model input features: "
    f"{len(MODEL_FEATURES):,}"
)


# ============================================================
# 5. GENERATE NEXT-QUARTER FORECASTS
# ============================================================

model_predictions_pd = champion_model.predict(
    model_input_pd
)

required_output_columns = {
    "predicted_next_quarter_transaction_count",
    "predicted_log_growth",
    "predicted_growth_pct"
}

missing_output_columns = sorted(
    required_output_columns
    - set(model_predictions_pd.columns)
)

assert not missing_output_columns, (
    f"Model output columns missing: "
    f"{missing_output_columns}"
)

assert (
    len(model_predictions_pd)
    == EXPECTED_STATE_COUNT
)

assert np.isfinite(
    model_predictions_pd[
        "predicted_next_quarter_transaction_count"
    ].to_numpy(dtype=float)
).all()

assert (
    model_predictions_pd[
        "predicted_next_quarter_transaction_count"
    ] > 0
).all()

print("Champion model scoring completed.")


# ============================================================
# 6. CREATE FORECAST OUTPUT
# ============================================================

forecast_pd = (
    scoring_pd[
        [
            "state_slug",
            "feature_period",
            "feature_period_start_date",
            "target_period",
            "target_period_start_date",
            "total_transaction_count"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

forecast_pd = forecast_pd.rename(
    columns={
        "total_transaction_count":
            "current_transaction_count"
    }
)

model_predictions_pd = (
    model_predictions_pd
    .reset_index(drop=True)
)

forecast_pd[
    "predicted_transaction_count"
] = (
    model_predictions_pd[
        "predicted_next_quarter_transaction_count"
    ]
    .round()
    .astype("int64")
)

forecast_pd[
    "predicted_log_growth"
] = (
    model_predictions_pd[
        "predicted_log_growth"
    ]
    .astype(float)
)

forecast_pd[
    "predicted_growth_pct"
] = (
    model_predictions_pd[
        "predicted_growth_pct"
    ]
    .astype(float)
)

forecast_pd[
    "absolute_predicted_change"
] = (
    forecast_pd[
        "predicted_transaction_count"
    ]
    - forecast_pd[
        "current_transaction_count"
    ]
)

forecast_pd[
    "naive_last_quarter_prediction"
] = forecast_pd[
    "current_transaction_count"
]

forecast_pd[
    "forecast_rank"
] = (
    forecast_pd[
        "predicted_transaction_count"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

forecast_pd[
    "model_name"
] = REGISTERED_MODEL_NAME

forecast_pd[
    "model_version"
] = MODEL_VERSION

forecast_pd[
    "model_alias"
] = MODEL_ALIAS

forecast_pd[
    "model_uri"
] = CHAMPION_MODEL_URI

forecast_pd[
    "source_feature_table"
] = SCORING_FEATURE_TABLE


# ============================================================
# 7. FORECAST QUALITY CHECKS
# ============================================================

duplicate_forecast_keys = (
    forecast_pd
    .duplicated(
        subset=[
            "state_slug",
            "target_period",
            "model_version"
        ]
    )
    .sum()
)

assert len(forecast_pd) == EXPECTED_STATE_COUNT

assert (
    forecast_pd["state_slug"].nunique()
    == EXPECTED_STATE_COUNT
)

assert forecast_pd["feature_period"].nunique() == 1
assert forecast_pd["target_period"].nunique() == 1
assert duplicate_forecast_keys == 0

assert (
    forecast_pd[
        "predicted_transaction_count"
    ] > 0
).all()

current_national_transactions = (
    forecast_pd[
        "current_transaction_count"
    ].sum()
)

predicted_national_transactions = (
    forecast_pd[
        "predicted_transaction_count"
    ].sum()
)

predicted_national_growth_pct = (
    100
    * (
        predicted_national_transactions
        / current_national_transactions
        - 1
    )
)

print("\n" + "=" * 70)
print("FORECAST SUMMARY")
print("=" * 70)

print(
    f"Feature period            : "
    f"{forecast_pd['feature_period'].iloc[0]}"
)

print(
    f"Forecast period           : "
    f"{forecast_pd['target_period'].iloc[0]}"
)

print(
    f"States/UTs forecast       : "
    f"{len(forecast_pd):,}"
)

print(
    f"Current national count    : "
    f"{current_national_transactions:,.0f}"
)

print(
    f"Predicted national count  : "
    f"{predicted_national_transactions:,.0f}"
)

print(
    f"Predicted national growth : "
    f"{predicted_national_growth_pct:.2f}%"
)

print(
    f"Registered model version  : "
    f"{MODEL_VERSION}"
)

print("=" * 70)


# ============================================================
# 8. SAVE FORECASTS TO DELTA
# ============================================================

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS workspace.ml"
)

forecast_spark_df = (
    spark.createDataFrame(forecast_pd)
    .withColumn(
        "scored_at_utc",
        F.current_timestamp()
    )
)

TEMPORARY_VIEW_NAME = (
    "phonepe_state_forecast_staging"
)

forecast_spark_df.createOrReplaceTempView(
    TEMPORARY_VIEW_NAME
)

if not spark.catalog.tableExists(FORECAST_TABLE):

    (
        forecast_spark_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(FORECAST_TABLE)
    )

    print(
        f"\nCreated forecast table: "
        f"{FORECAST_TABLE}"
    )

else:

    spark.sql(
        f"""
        merge into {FORECAST_TABLE} as target

        using {TEMPORARY_VIEW_NAME} as source

        on
            target.state_slug = source.state_slug
            and target.target_period = source.target_period
            and target.model_version = source.model_version

        when matched then update set *

        when not matched then insert *
        """
    )

    print(
        f"\nUpdated forecast table: "
        f"{FORECAST_TABLE}"
    )


# ============================================================
# 9. VALIDATE SAVED FORECASTS
# ============================================================

saved_forecasts_df = (
    spark.table(FORECAST_TABLE)
    .filter(
        (
            F.col("target_period")
            == forecast_pd[
                "target_period"
            ].iloc[0]
        )
        &
        (
            F.col("model_version")
            == MODEL_VERSION
        )
    )
)

saved_summary = (
    saved_forecasts_df
    .agg(
        F.count("*").alias("saved_rows"),

        F.countDistinct(
            "state_slug"
        ).alias("saved_states"),

        F.sum(
            "predicted_transaction_count"
        ).alias(
            "predicted_national_transaction_count"
        ),

        F.min(
            "scored_at_utc"
        ).alias("first_scored_at_utc"),

        F.max(
            "scored_at_utc"
        ).alias("last_scored_at_utc")
    )
    .first()
)

saved_duplicate_keys = (
    saved_forecasts_df
    .groupBy(
        "state_slug",
        "target_period",
        "model_version"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

assert (
    saved_summary["saved_rows"]
    == EXPECTED_STATE_COUNT
)

assert (
    saved_summary["saved_states"]
    == EXPECTED_STATE_COUNT
)

assert saved_duplicate_keys == 0

print("\nSaved forecast validation")

print(
    f"Saved rows              : "
    f"{saved_summary['saved_rows']:,}"
)

print(
    f"States/UTs represented  : "
    f"{saved_summary['saved_states']:,}"
)

print(
    f"Predicted national count: "
    f"{saved_summary['predicted_national_transaction_count']:,}"
)

print(
    f"Duplicate forecast keys : "
    f"{saved_duplicate_keys:,}"
)

print(
    "Final production forecast validation passed."
)


# ============================================================
# 10. DISPLAY FORECASTS
# ============================================================

display(
    saved_forecasts_df
    .select(
        "forecast_rank",
        "state_slug",
        "feature_period",
        "target_period",
        "current_transaction_count",
        "predicted_transaction_count",
        "predicted_growth_pct",
        "absolute_predicted_change",
        "model_version",
        "scored_at_utc"
    )
    .orderBy(
        "forecast_rank"
    )
)

PHONEPE STATE TRANSACTION BATCH SCORING
Feature table : workspace.gold.mart_ml_state_scoring_features
Model         : workspace.ml.phonepe_state_transaction_forecaster
Model alias   : Champion
Forecast table: workspace.ml.state_transaction_forecasts


{"ts": "2026-10-11 20:23:14.992", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjAyNDUzNDY3MTQyODM5NBABIAEyJDAxYTEyY2EyLTA0OTgtNzIzNi04ZmQ0LTU0YmY5NWM0ZGI3ZTokMzhlZDM2MGItZGNhMS0zZmFmLWJhNDUtZmQzNjczMTNkNGY3SgwIoeCv1gYQgJiRwgNQAVgBYAFoo7Wzsv7Eow2IAQA=.", "context": {}}
{"ts": "2026-10-11 20:23:14.992", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjAyNDUzNDY3MTQyODM5NBABIAEyJDAxYTEyY2EyLTA0OTgtNzIzNi04ZmQ0LTU0YmY5NWM0ZGI3ZTokMzhlZDM2MGItZGNhMS0zZmFmLWJhNDUtZmQzNjczMTNkNGY3SgwIoeCv1gYQgJiRwgNQAVgBYAFoo7Wzsv7Eow2IAQA=.", "context": {}}
{"ts": "2026-10-11 20:23:14.992", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMjAyNDUzNDY3MTQyODM5NBABIAEyJDAxYTEyY2EyLTA0OTgtNzIzNi04ZmQ0LTU0YmY5NWM0ZGI3ZTokMzhlZDM2MGItZGNhMS0zZmFmLWJhNDUtZmQzNjcz


Scoring feature validation
Rows          : 36
States/UTs    : 36
Feature period: 2026-Q2
Target period : 2026-Q3
Missing values: 0
Scoring feature validation passed.



Champion model loaded
Model URI    : models:/workspace.ml.phonepe_state_transaction_forecaster@Champion
Model version: 1

Model input validation passed.
Model input rows    : 36
Model input features: 22
Champion model scoring completed.

FORECAST SUMMARY
Feature period            : 2026-Q2
Forecast period           : 2026-Q3
States/UTs forecast       : 36
Current national count    : 38,664,285,691
Predicted national count  : 39,395,063,445
Predicted national growth : 1.89%
Registered model version  : 1

Created forecast table: workspace.ml.state_transaction_forecasts

Saved forecast validation
Saved rows              : 36
States/UTs represented  : 36
Predicted national count: 39,395,063,445
Duplicate forecast keys : 0
Final production forecast validation passed.


forecast_rank,state_slug,feature_period,target_period,current_transaction_count,predicted_transaction_count,predicted_growth_pct,absolute_predicted_change,model_version,scored_at_utc
1,maharashtra,2026-Q2,2026-Q3,5064451991,5097045776,0.6435797001113963,32593785,1,2026-10-11T20:23:51.578Z
2,karnataka,2026-Q2,2026-Q3,4491711923,4546892420,1.2284959053506128,55180497,1,2026-10-11T20:23:51.578Z
3,telangana,2026-Q2,2026-Q3,3878232440,3967873985,2.3114020796357826,89641545,1,2026-10-11T20:23:51.578Z
4,uttar-pradesh,2026-Q2,2026-Q3,3665306240,3671109331,0.1583248491094258,5803091,1,2026-10-11T20:23:51.578Z
5,andhra-pradesh,2026-Q2,2026-Q3,3245956863,3190676067,-1.7030662544059605,-55280796,1,2026-10-11T20:23:51.578Z
6,rajasthan,2026-Q2,2026-Q3,2553428529,2642330265,3.481661436311745,88901736,1,2026-10-11T20:23:51.578Z
7,madhya-pradesh,2026-Q2,2026-Q3,2283838085,2295489939,0.5101874139581852,11651854,1,2026-10-11T20:23:51.578Z
8,bihar,2026-Q2,2026-Q3,2289051873,2295075372,0.263143853446679,6023499,1,2026-10-11T20:23:51.578Z
9,west-bengal,2026-Q2,2026-Q3,1782353009,1791654056,0.5218409071860108,9301047,1,2026-10-11T20:23:51.578Z
10,odisha,2026-Q2,2026-Q3,1523130010,1561655217,2.529344609812423,38525207,1,2026-10-11T20:23:51.578Z
